In [1]:
!pip install jiwer -q

import nltk
nltk.download('punkt', quiet=True)

True

In [2]:
#IMPORTS
import os
import re
import cv2
import time
import math
import torch
import pickle
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
 
from jiwer import wer
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
#Paths
train_i3d_path      = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/i3d_features_rwth phoenix 2014t/i3d_features_rwth phoenix 2014t/train"
train_mediapipe_path= "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/mediapipe_features_rwth phoenix weather 2014t/mediapipe_features/train"
val_i3d_path        = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/i3d_features_rwth phoenix 2014t/i3d_features_rwth phoenix 2014t/val"
val_mediapipe_path  = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/mediapipe_features_rwth phoenix weather 2014t/mediapipe_features/val"
train_tsv_path      = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/tsv files_rwth phoenix 2014t/tsv files/cvpr23.fairseq.i3d.train.how2sign.tsv"
val_tsv_path        = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/tsv files_rwth phoenix 2014t/tsv files/cvpr23.fairseq.i3d.val.how2sign.tsv"
 

In [4]:
#DATASET
class CSLRDataset(Dataset):
 
    def __init__(self, i3d_path, mediapipe_path, df,
                 text_col="translation", target_len=128, augment=False):
 
        self.i3d_path       = i3d_path
        self.mediapipe_path = mediapipe_path
        self.target_len     = target_len
        self.augment        = augment
 
        self.label_map = dict(zip(df["id"], df[text_col]))
 
        i3d_files    = set(os.listdir(i3d_path))
        mp_files     = set(os.listdir(mediapipe_path))
        common_files = sorted(list(i3d_files & mp_files))
 
        self.files = [
            f for f in common_files
            if os.path.splitext(f)[0] in self.label_map
        ]
        print(f"Matched common files: {len(self.files)}")
 
    def __len__(self):
        return len(self.files)
 
    def temporal_resize(self, features):
        return cv2.resize(
            features,
            (features.shape[1], self.target_len),
            interpolation=cv2.INTER_LINEAR
        )
 
    # ── IMPROVEMENT 1: Stronger augmentation ──────────────
    def augment_features(self, x):
 
        # (a) Gaussian noise — slightly stronger than before (0.015 vs 0.01)
        if np.random.rand() < 0.8:
            noise = np.random.normal(0, 0.015, x.shape)
            x = x + noise
 
        # (b) TIME STRETCH — most impactful new addition
        #     Simulates different signing speeds (0.75x to 1.25x)
        if np.random.rand() < 0.5:
            factor  = np.random.uniform(0.75, 1.25)
            new_T   = max(10, int(x.shape[0] * factor))
            indices = np.linspace(0, x.shape[0] - 1, new_T).astype(int)
            x       = x[indices]
            x       = self.temporal_resize(x)   # back to TARGET_LEN
 
        # (c) FEATURE MASKING — randomly zero out 10% of feature dims
        #     Teaches model to be robust to missing keypoints
        if np.random.rand() < 0.3:
            mask_cols = int(x.shape[1] * 0.10)
            start     = np.random.randint(0, x.shape[1] - mask_cols)
            x[:, start:start + mask_cols] = 0
 
        # (d) TEMPORAL MASKING — zero out a random time segment
        #     Like SpecAugment but for video features
        if np.random.rand() < 0.3:
            t_start = np.random.randint(0, max(1, x.shape[0] - 15))
            t_end   = min(x.shape[0], t_start + np.random.randint(5, 15))
            x[t_start:t_end, :] = 0
 
        # (e) Random frame drop (kept from original)
        if np.random.rand() < 0.3:
            keep_ratio = np.random.uniform(0.85, 0.95)
            keep_len   = max(10, int(len(x) * keep_ratio))
            idx        = np.sort(np.random.choice(len(x), keep_len, replace=False))
            x          = x[idx]
            x          = self.temporal_resize(x)
 
        return x
 
    def __getitem__(self, idx):
        fname = self.files[idx]
 
        i3d = np.load(os.path.join(self.i3d_path, fname))
        mp  = np.load(os.path.join(self.mediapipe_path, fname))
        mp  = mp.reshape(mp.shape[0], -1)
 
        i3d = self.temporal_resize(i3d)
        mp  = self.temporal_resize(mp)
 
        x = np.concatenate([i3d, mp], axis=1)
 
        if self.augment:
            x = self.augment_features(x)
 
        x          = torch.tensor(x, dtype=torch.float32)
        video_id   = os.path.splitext(fname)[0]
        sentence   = self.label_map[video_id]
 
        return x, sentence
 


In [5]:
#COLLATE
def collate_fn(batch):
    features  = [item[0] for item in batch]
    sentences = [item[1] for item in batch]
    features  = pad_sequence(features, batch_first=True)
    return features, sentences

In [6]:
#VOCABULARY
train_df = pd.read_csv(train_tsv_path, sep="\t")
val_df   = pd.read_csv(val_tsv_path,   sep="\t")
 
vocab = set()
for sentence in train_df["translation"]:
    vocab.update(sentence.lower().split())
 
vocab    = ["<blank>", "<sos>", "<eos>"] + sorted(list(vocab))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
 
SOS_IDX   = word2idx["<sos>"]
EOS_IDX   = word2idx["<eos>"]
BLANK_IDX = word2idx["<blank>"]
 
print("Vocabulary size:", len(vocab))
print("SOS:", SOS_IDX, "| EOS:", EOS_IDX)

Vocabulary size: 2890
SOS: 1 | EOS: 2


In [7]:
#DATASETS & LOADERS
train_dataset = CSLRDataset(
    i3d_path=train_i3d_path,
    mediapipe_path=train_mediapipe_path,
    df=train_df,
    text_col="translation",
    augment=True                        # augmentation ON for training
)
 
val_dataset = CSLRDataset(
    i3d_path=val_i3d_path,
    mediapipe_path=val_mediapipe_path,
    df=val_df,
    text_col="translation",
    augment=False                       # NO augmentation for validation
)
 
loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
 
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
 
print(f"Train: {len(train_dataset)} samples | Val: {len(val_dataset)} samples")
 

Matched common files: 7096
Matched common files: 519
Train: 7096 samples | Val: 519 samples


In [8]:
#MODEL
class MultiScaleCNN(nn.Module):
 
    def __init__(self, channels):
        super().__init__()
        self.branch3 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.branch5 = nn.Conv1d(channels, channels, kernel_size=5, padding=2)
        self.branch7 = nn.Conv1d(channels, channels, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm1d(channels * 3)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
 
    def forward(self, x):
        x = torch.cat([self.branch3(x), self.branch5(x), self.branch7(x)], dim=1)
        return self.dropout(self.relu(self.bn(x)))
 
 
class CSLRModel(nn.Module):
 
    def __init__(self, vocab_size, d_model=256, nhead=8,
                 num_decoder_layers=4, max_len=128):
        super().__init__()
        self.d_model = d_model
 
        # Encoder
        self.proj = nn.Linear(1123, 512)
        self.conv = MultiScaleCNN(512)
        self.lstm = nn.LSTM(1536, d_model, num_layers=2,
                                    batch_first=True, bidirectional=True, dropout=0.3)
        self.encoder_proj = nn.Linear(d_model * 2, d_model)
        self.encoder_norm = nn.LayerNorm(d_model)
 
        # Decoder
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = nn.Embedding(max_len + 2, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=1024, dropout=0.3, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
 
    def encode(self, x):
        x = self.proj(x)
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = self.encoder_norm(self.encoder_proj(x))
        return x
 
    def decode(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        B, S = tgt.shape
        pos = torch.arange(S, device=tgt.device).unsqueeze(0).expand(B, -1)
        tgt_emb = self.embedding(tgt) + self.pos_encoding(pos)
        out = self.decoder(tgt=tgt_emb, memory=memory,tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.fc_out(out)
 
    def forward(self, x, tgt):
        memory = self.encode(x)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.shape[1]).to(x.device)
        return self.decode(tgt, memory, tgt_mask=tgt_mask)
 
 
model = CSLRModel(
    vocab_size=len(vocab),
    d_model=256,
    nhead=8,
    num_decoder_layers=4,
    max_len=128
).to(device)
 
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")
 


Model parameters: 15,624,778


In [9]:
#HELPERS
def encode_sentence(sentence, word2idx):
    tokens  = sentence.lower().split()
    encoded = [word2idx[w] for w in tokens if w in word2idx]
    return [SOS_IDX] + encoded + [EOS_IDX]
 
 
# Greedy decode (used during training WER checks — fast)
def generate_sentence(model, features, idx2word, word2idx, max_len=50):
    model.eval()
    features  = features.unsqueeze(0).to(device)
    generated = [SOS_IDX]
 
    with torch.no_grad():
        memory = model.encode(features)
        for _ in range(max_len):
            tgt = torch.tensor([generated], dtype=torch.long).to(device)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.shape[1]).to(device)
            
            output = model.decode(tgt, memory, tgt_mask=tgt_mask)
            next_token = output[:, -1, :].argmax(-1).item()
            
            if next_token == EOS_IDX:
                break
            generated.append(next_token)
 
    words = [idx2word[i] for i in generated[1:] if i in idx2word]
    return " ".join(words)
 
 
# ── IMPROVEMENT 2: Warmup + Cosine LR scheduler 
# Original used ReduceLROnPlateau (reactive).
# Warmup + Cosine is proactive: LR ramps up for 5 epochs,
# then smoothly decays. Much better for transformers.

def get_lr(optimizer):
    return optimizer.param_groups[0]['lr']
 
WARMUP_EPOCHS = 5
NUM_EPOCHS    = 60
 
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
 
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
 
# Warmup scheduler (linear ramp for first WARMUP_EPOCHS)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR( optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)

# Cosine decay for the rest

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR( optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-7)
 
 
# ── IMPROVEMENT 3: Scheduled Sampling ratio 
# During training, instead of always feeding the GROUND TRUTH
# previous token to the decoder (teacher forcing), we
# occasionally feed the MODEL'S OWN prediction.
# This directly fixes the "haben wir haben wir" repetition
# because the model learns to recover from its own mistakes.
# Ratio starts at 0 (pure teacher forcing) and rises to 0.3.

def get_ss_ratio(epoch, max_ratio=0.30, ramp_epochs=30):
    return min(max_ratio, (epoch / ramp_epochs) * max_ratio)
 


In [10]:
#TRAINING
ACCUMULATION_STEPS = 2      # IMPROVEMENT 4: gradient accumulation
                             # effective batch = 8 * 2 = 16
PATIENCE = 15
best_val_loss = float("inf")
patience_counter = 0
total_start = time.time()
 
for epoch in range(NUM_EPOCHS):
    epoch_start  = time.time()
    ss_ratio = get_ss_ratio(epoch)      # scheduled sampling ratio
 
    # Train 
    model.train()
    total_loss = 0
    valid_batches = 0
    optimizer.zero_grad()
 
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"LR: {get_lr(optimizer):.2e} | "
          f"SS ratio: {ss_ratio:.3f} =====")
 
    for batch_idx, (features, sentences) in enumerate(loader):
 
        features = features.to(device)
 
        encoded_sentences = [
            torch.tensor(encode_sentence(s, word2idx), dtype=torch.long)
            for s in sentences
        ]
 
        targets = pad_sequence(
            encoded_sentences, batch_first=True, padding_value=0
        ).to(device)
 
        decoder_input  = targets[:, :-1]   # teacher-forced input
        decoder_target = targets[:, 1:]    # expected output
 
        # IMPROVEMENT 3 applied: scheduled sampling 
        # With probability ss_ratio, replace decoder input tokens
        # (after position 1) with the model's own greedy predictions.
        # This is done token-by-token in a fast batch manner.
        
        if ss_ratio > 0 and random.random() < ss_ratio:
            with torch.no_grad():
                memory   = model.encode(features)
                
                # build a mixed input: start with ground truth SOS,
                # then greedily generate the rest
                
                mixed_input = decoder_input.clone()
                for t in range(1, decoder_input.shape[1]):
                    tgt_so_far = mixed_input[:, :t]
                    tgt_mask   = nn.Transformer.generate_square_subsequent_mask(t).to(device)
                    logits = model.decode(tgt_so_far, memory, tgt_mask=tgt_mask)
                    pred_token = logits[:, -1, :].argmax(-1)  # (B,)
                    
                    # replace with model prediction with probability ss_ratio
                    use_pred   = (torch.rand(pred_token.shape, device=device) < ss_ratio)
                    mixed_input[:, t] = torch.where(
                        use_pred, pred_token, decoder_input[:, t]
                    )
            decoder_input = mixed_input
 
        output = model(features, decoder_input)
 
        output = output.reshape(-1, output.shape[-1])
        decoder_target = decoder_target.reshape(-1)
 
        loss = criterion(output, decoder_target) / ACCUMULATION_STEPS
        loss.backward()
 
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
 
        total_loss    += loss.item() * ACCUMULATION_STEPS
        valid_batches += 1
 
        if batch_idx % 50 == 0:
            print(f"  Batch {batch_idx} | Loss: {loss.item() * ACCUMULATION_STEPS:.4f}")
 
    avg_train_loss = total_loss / max(valid_batches, 1)
    print(f"\nEpoch {epoch+1} | Avg Train Loss: {avg_train_loss:.4f}")
 
    # Validation loss 
    model.eval()
    val_loss_total = 0
    val_batches = 0
 
    with torch.no_grad():
        for features, sentences in val_loader:
            features = features.to(device)
 
            encoded_sentences = [
                torch.tensor(encode_sentence(s, word2idx), dtype=torch.long)
                for s in sentences
            ]
            targets = pad_sequence(
                encoded_sentences, batch_first=True, padding_value=0
            ).to(device)
 
            decoder_input  = targets[:, :-1]
            decoder_target = targets[:, 1:]
 
            output = model(features, decoder_input)
            output = output.reshape(-1, output.shape[-1])
            decoder_target = decoder_target.reshape(-1)
 
            loss = criterion(output, decoder_target)
            val_loss_total += loss.item()
            val_batches += 1
 
    avg_val_loss = val_loss_total / max(val_batches, 1)
    print(f"Validation Loss: {avg_val_loss:.4f}")
 
    #  IMPROVEMENT 5: WER check every 5 epochs 
    # Lets you monitor actual translation quality during training,
    # not just loss (loss going down doesn't always mean WER improves).
    if (epoch + 1) % 5 == 0:
        refs, hyps = [], []
        sample_count = 0
        with torch.no_grad():
            for features, sentences in val_loader:
                features = features.to(device)
                for i in range(features.shape[0]):
                    pred = generate_sentence(model, features[i], idx2word, word2idx)
                    refs.append(sentences[i].lower())
                    hyps.append(pred.lower())
                    sample_count += 1
                    if sample_count >= 200:   # fast check on 200 samples
                        break
                if sample_count >= 200:
                    break
        mid_wer = wer(refs, hyps)
        print(f"  ↳ Mid-training WER (200 samples): {mid_wer:.4f}")
 
    #  Save best 
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "seq2seq_best.pth")
        patience_counter = 0
        print("✅ Best model saved")
    else:
        patience_counter += 1
        print(f"No improvement for {patience_counter} epochs")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered")
            break
 
    #  LR schedule 
    if epoch < WARMUP_EPOCHS:
        warmup_scheduler.step()
    else:
        cosine_scheduler.step()
 
    print(f"⏱️  Epoch time: {time.time() - epoch_start:.1f}s")
 
print(f"\nTotal training time: {(time.time() - total_start)/60:.1f} min")
 
 


===== Epoch 1/60 | LR: 1.00e-05 | SS ratio: 0.000 =====
  Batch 0 | Loss: 8.0410
  Batch 50 | Loss: 7.7068
  Batch 100 | Loss: 7.6103
  Batch 150 | Loss: 7.4498
  Batch 200 | Loss: 7.2449
  Batch 250 | Loss: 7.3599
  Batch 300 | Loss: 7.1401
  Batch 350 | Loss: 7.0756
  Batch 400 | Loss: 6.9329
  Batch 450 | Loss: 7.0660
  Batch 500 | Loss: 6.9016
  Batch 550 | Loss: 6.7476
  Batch 600 | Loss: 7.0530
  Batch 650 | Loss: 6.7025
  Batch 700 | Loss: 6.8515
  Batch 750 | Loss: 6.4861
  Batch 800 | Loss: 6.7815
  Batch 850 | Loss: 6.7569

Epoch 1 | Avg Train Loss: 7.0365
Validation Loss: 6.4333
✅ Best model saved
⏱️  Epoch time: 113.7s

===== Epoch 2/60 | LR: 2.80e-05 | SS ratio: 0.010 =====
  Batch 0 | Loss: 6.6440
  Batch 50 | Loss: 6.3643
  Batch 100 | Loss: 6.2689
  Batch 150 | Loss: 6.4057
  Batch 200 | Loss: 6.2460
  Batch 250 | Loss: 6.1528
  Batch 300 | Loss: 6.2225
  Batch 350 | Loss: 6.2068
  Batch 400 | Loss: 6.0632
  Batch 450 | Loss: 6.2055
  Batch 500 | Loss: 5.6838
  Batch 5

In [11]:
# Load best & evaluate
model.load_state_dict(torch.load("seq2seq_best.pth"))
model.eval()
print("✅ Best model loaded")


✅ Best model loaded


In [12]:
# Full WER evaluation
 
references, hypotheses = [], []
 
with torch.no_grad():
    for features, sentences in val_loader:
        features = features.to(device)
        for i in range(features.shape[0]):
            pred = generate_sentence(model, features[i], idx2word, word2idx)
            references.append(sentences[i].lower())
            hypotheses.append(pred.lower())
 
greedy_wer = wer(references, hypotheses)
print(f"\nFINAL GREEDY WER: {greedy_wer:.4f}")


FINAL GREEDY WER: 0.8021


In [13]:
# CELL 12: Improved beam search 
# Changes vs original:
#   (a) Repetition penalty — discourages repeating recent tokens
#   (b) Proper EOS handling — only truly-completed beams win
#   (c) Alpha=0.7 — rewards longer sequences slightly more
def beam_search_generate(model, features, idx2word, beam_width=5, max_len=50, alpha=0.7):
    model.eval()
    features = features.unsqueeze(0).to(device)
 
    with torch.no_grad():
        memory = model.encode(features)
        beams = [([SOS_IDX], 0.0)]
        completed = []
 
        for _ in range(max_len):
            new_beams = []
 
            for tokens, score in beams:
                if tokens[-1] == EOS_IDX:
                    completed.append((tokens, score))
                    continue
 
                tgt      = torch.tensor([tokens], dtype=torch.long).to(device)
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                    tgt.shape[1]).to(device)
                output = model.decode(tgt, memory, tgt_mask=tgt_mask)
                logits = output[:, -1, :]
                probs = F.log_softmax(logits, dim=-1)
                topk  = torch.topk(probs, beam_width)
 
                recent_tokens = set(tokens[-6:])   # last 6 tokens
 
                for i in range(beam_width):
                    next_token = topk.indices[0][i].item()
                    next_score = score + topk.values[0][i].item()
 
                    # (a) REPETITION PENALTY
                    if next_token in recent_tokens:
                        next_score -= 0.4
 
                    new_beams.append((tokens + [next_token], next_score))
 
            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
 
        completed.extend(beams)
 
        # (b) prefer beams that actually hit EOS
        truly_done = [b for b in completed if b[0][-1] == EOS_IDX]
        pool = truly_done if truly_done else completed
 
        # (c) length-normalised score with alpha=0.7
        best_tokens = sorted(
            pool,
            key=lambda x: x[1] / (len(x[0]) ** alpha),
            reverse=True
        )[0][0]
 
        words = [
            idx2word[idx] for idx in best_tokens
            if idx not in (SOS_IDX, EOS_IDX, BLANK_IDX) and idx in idx2word
        ]
        return " ".join(words)
 

In [14]:
# CELL 13: Improved post-processing 
def fix_common_errors(text):
    words = text.split()
    cleaned = []
    for w in words:
        if cleaned and cleaned[-1] == w:
            continue
        cleaned.append(w)
    text = " ".join(cleaned)
    # remove repeated 2-word and 3-word phrases
    text = re.sub(r'\b(\w+ \w+)( \1)+\b', r'\1', text)
    text = re.sub(r'\b(\w+ \w+ \w+)( \1)+\b', r'\1', text)
    return text

In [15]:
#  CELL 14: Final beam WER 
refs_b, hyps_b = [], []
 
with torch.no_grad():
    for features, sentences in val_loader:
        features = features.to(device)
        for i in range(features.shape[0]):
            pred = beam_search_generate(model, features[i], idx2word, beam_width=5)
            pred = fix_common_errors(pred)
            refs_b.append(sentences[i].lower())
            hyps_b.append(pred.lower())
 
beam_wer = wer(refs_b, hyps_b)
print(f"\nFINAL BEAM SEARCH WER: {beam_wer:.4f}")



FINAL BEAM SEARCH WER: 0.7553


In [16]:
# CELL 15: BLEU scores 
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
 
smoothie = SmoothingFunction().method1
refs_nl  = [[r.split()] for r in refs_b]
hyps_nl  = [h.split()   for h in hyps_b]
 
b1 = corpus_bleu(refs_nl, hyps_nl, weights=(1,0,0,0),smoothing_function=smoothie)
b2 = corpus_bleu(refs_nl, hyps_nl, weights=(0.5,0.5,0,0),smoothing_function=smoothie)
b3 = corpus_bleu(refs_nl, hyps_nl, weights=(0.33,0.33,0.33,0),smoothing_function=smoothie)
b4 = corpus_bleu(refs_nl, hyps_nl, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie)
 
print(f"BLEU-1: {b1:.4f}")
print(f"BLEU-2: {b2:.4f}")
print(f"BLEU-3: {b3:.4f}")
print(f"BLEU-4: {b4:.4f}")


BLEU-1: 0.3684
BLEU-2: 0.2674
BLEU-3: 0.2071
BLEU-4: 0.1624


In [17]:
#  CELL 16: Save everything 
torch.save(model.state_dict(), "seq2seq_best.pth")
 
with open("word2idx.pkl", "wb") as f: pickle.dump(word2idx, f)
with open("idx2word.pkl", "wb") as f: pickle.dump(idx2word, f)
with open("vocab.pkl", "wb") as f: pickle.dump(vocab,f)
 
# Config now includes sos_idx / eos_idx so prediction notebook loads correctly
config = {
    "vocab_size":len(vocab),
    "d_model": 256,
    "nhead": 8,
    "num_decoder_layers": 4,
    "max_len": 128,
    "beam_width": 5,
    "sos_idx": SOS_IDX,
    "eos_idx": EOS_IDX,
    "pad_idx": BLANK_IDX,
}
with open("config.pkl", "wb") as f: pickle.dump(config, f)
 
print("✅ All files saved")
print(f"\nSummary:")
print(f" Greedy WER : {greedy_wer:.4f}")
print(f" Beam WER : {beam_wer:.4f}")
print(f"BLEU-4 : {b4:.4f}")

✅ All files saved

Summary:
 Greedy WER : 0.8021
 Beam WER : 0.7553
BLEU-4 : 0.1624


In [24]:
# =========================================
# RANDOM PREDICTION TEST
# =========================================

import random
import torch
import torch.nn.functional as F
import re


# CLEAN REPETITIONS

def fix_common_errors(text):

    # remove repeated words
    text = re.sub(
        r'\b(\w+)( \1){2,}\b',
        r'\1',
        text
    )

    # remove repeated 2-word phrases
    text = re.sub(
        r'\b(\w+ \w+)( \1)+\b',
        r'\1',
        text
    )

    # remove repeated 3-word phrases
    text = re.sub(
        r'\b(\w+ \w+ \w+)( \1)+\b',
        r'\1',
        text
    )

    return text


# BEAM SEARCH GENERATION

def beam_search_generate(
    model,
    features,
    idx2word,
    beam_width=5,
    max_len=50,
    alpha=0.7
):

    model.eval()

    features = features.unsqueeze(0).to(device)

    with torch.no_grad():

        memory = model.encode(features)

        beams = [
            ([SOS_IDX], 0.0)
        ]

        completed = []

        for _ in range(max_len):

            new_beams = []

            for tokens, score in beams:

                if tokens[-1] == EOS_IDX:

                    completed.append(
                        (tokens, score)
                    )

                    continue

                tgt = torch.tensor(
                    [tokens],
                    dtype=torch.long
                ).to(device)

                tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                    tgt.shape[1]
                ).to(device)

                output = model.decode(
                    tgt,
                    memory,
                    tgt_mask=tgt_mask
                )

                logits = output[:, -1, :]

                probs = F.log_softmax(
                    logits,
                    dim=-1
                )

                # repetition penalty
                repetition_penalty = 1.3

                for token_id in set(tokens):

                    if token_id not in [
                        SOS_IDX,
                        EOS_IDX,
                        BLANK_IDX
                    ]:

                        probs[0][token_id] /= repetition_penalty

                topk = torch.topk(
                    probs,
                    beam_width
                )

                for i in range(beam_width):

                    next_token = topk.indices[0][i].item()

                    next_score = (
                        score
                        + topk.values[0][i].item()
                    )

                    new_beams.append(
                        (
                            tokens + [next_token],
                            next_score
                        )
                    )

            beams = sorted(
                new_beams,
                key=lambda x: x[1],
                reverse=True
            )[:beam_width]

        completed.extend(beams)

        truly_completed = [

            b for b in completed

            if b[0][-1] == EOS_IDX
        ]

        if not truly_completed:

            truly_completed = completed

        best_tokens = sorted(

            truly_completed,

            key=lambda x: (
                x[1]
                / (len(x[0]) ** alpha)
            ),

            reverse=True

        )[0][0]

        words = []

        for idx in best_tokens:

            if idx in [
                SOS_IDX,
                EOS_IDX,
                BLANK_IDX
            ]:
                continue

            if idx in idx2word:

                words.append(
                    idx2word[idx]
                )

        sentence = " ".join(words)

        sentence = fix_common_errors(
            sentence
        )

        return sentence


# RANDOM VALIDATION SAMPLE

idx = random.randint(
    0,
    len(val_dataset) - 1
)

features, actual_sentence = val_dataset[idx]

predicted_sentence = beam_search_generate(
    model,
    features,
    idx2word,
    beam_width=5
)

print("\n========== RESULT ==========")

print("\nACTUAL:")
print(actual_sentence)

print("\nPREDICTED:")
print(predicted_sentence)


========== RESULT ==========

ACTUAL:
heute nacht temperaturen zwischen minus drei und minus siebzehn grad wo es länger aufklart werte darunter

PREDICTED:
heute nacht werte zwischen minus drei und minus siebzehn grad im laufe des tages minus siebzehn grad
